# EXP0008 — Vision Transformer ViT-B/16 (`MODEL0004`) Kaggle Training + Cross-Dataset Evaluation

**One-session turnkey notebook (~70–85 min on Tesla T4).**

### Notebook Settings (Right Sidebar):
1. **Accelerator:** `GPU T4 x2` (do NOT select P100)
2. **Internet:** `On`
3. **Inputs (`+ Add Input`):**
   - `isthisdeception/ds0001-pp0001` *(required — training & in-distribution test)*
   - `isthisdeception/ds0002-pp0002` *(recommended — unseen DiFF diffusion faces)*
   - `isthisdeception/ds0003-pp0003` *(recommended — 140k Real & Fake Faces)*
   - `isthisdeception/ds0004-pp0006` *(recommended — Synthbuster)*
   - `isthisdeception/ds0005-pp0005` *(recommended — FairFace demographic real portraits)*
4. **Optional Secrets (`Add-ons -> Secrets`):** `GITHUB_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY` *(not required if downloading `/kaggle/working/EXP0008_results_bundle.zip` directly!)*

## 0. Setup & Environment Bootstrap

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/isthisdeception/thesis.git'
BRANCH = 'main'
REPO = Path('/kaggle/working/thesis')
EXP = 'EXP0008'
EVAL_ID = 'EVAL0010'
MODEL_ID = 'MODEL0004'
AUTOMATION = REPO / '17_Automation'
CONFIG = REPO / '06_Experiments' / EXP / 'config.yaml'
PLAN = 'kaggle_sync/sync_plan.exp0008.yaml'

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plum-dispatch==2.5.7'], check=True)
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
    import timm

print('Git HEAD:', subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
print('timm version:', timm.__version__)

In [ ]:
# Self-bootstrap EXP0008 config & registry rows if running before pushing local branch to origin/main
CONFIG_YAML = '''schema_version: 1
experiment_id: EXP0008
research_question: How can we improve the cross-generator generalization of image-based AI face detection models to maintain high accuracy on unseen generative architectures?
purpose: Evaluate patch-based Vision Transformer MODEL0004 (ViT-B/16 via timm) under the exact EXP0001 fairness contract on DS0001 / v1.0 / PP0001 / SPLIT0001 (224x224).
expected_outcome: Distinct global self-attention representation relative to CNN baselines on SPLIT0001 and external corpora, enabling cross-architecture contested abstention in M7.
study_track: primary_architecture
dataset:
  dataset_id: DS0001
  version: v1.0
  pipeline_id: PP0001
  output_id: DS0001_PP0001
  processed_kaggle_slug: isthisdeception/ds0001-pp0001
  processed_root: /kaggle/input/datasets/isthisdeception/ds0001-pp0001/DS0001_PP0001
  path_column: processed_path
  label_column: class_label
  partition_column: partition
  vocab: [fake, real]
split:
  split_id: SPLIT0001
  scheme: grouped_random
  assignments_csv: 03_Datasets/splits/DS0001_PP0001_SPLIT0001/assignments.csv
  train_name: train
  valid_name: val
  test_name: test
  leakage_required: PASS
model:
  model_id: MODEL0004
  architecture: ViT-B/16
  backbone: vit_base_patch16_224.augreg2_in21k_ft_in1k
  source: timm
  pretrained: true
  pretrained_weights: ImageNet-21k-ft-1k
fastai:
  datablock:
    blocks: [ImageBlock, CategoryBlock]
    get_items: assignments_csv
    get_x: processed_root + processed_path
    get_y: class_label
    splitter: column_splitter
    n_inp: 1
  item_tfms:
    resize: 224
    method: squish
  normalize:
    stats: imagenet
    mean: [0.485, 0.456, 0.406]
    std: [0.229, 0.224, 0.225]
  dataloaders:
    batch_size_eval: 128
    num_workers: 2
    drop_last: false
    shuffle_train: true
  mixed_precision: true
  lr_finder:
    enabled: true
    suggest_func: valley
    num_it: 100
  schedule:
    method: fine_tune
    freeze_epochs: 1
    lr_mult: 100
    pct_start: 0.3
    div: 5.0
  callbacks:
    save_model:
      monitor: valid_loss
      comp: min
      every_epoch: true
      with_opt: true
    early_stopping:
      enabled: true
      monitor: valid_loss
      patience: 3
    csv_logger: true
  export:
    export_pkl: true
    weights_pth: true
image_size: 224
batch_size: 64
epochs: 10
optimizer:
  name: Adam
  weight_decay: 0.01
  betas: [0.9, 0.99]
learning_rate:
  value: 2.0e-4
  source: manual
loss_function:
  name: CrossEntropyLossFlat
  label_smoothing: 0.0
metrics:
  primary: [roc_auc, pr_auc, f1, balanced_accuracy, mcc, precision, recall, specificity, fpr, fnr]
  secondary: [accuracy, ece, brier]
  monitor: valid_loss
  positive_class: fake
  threshold_policy: validation_only
  threshold_criterion: max_balanced_accuracy
  calibration_bins: 15
random_seed: 42
augmentations:
  aug_transforms:
    enabled: true
    train_only: true
    do_flip: true
    flip_vert: false
    max_rotate: 10.0
    max_zoom: 1.05
    max_lighting: 0.1
    max_warp: 0.0
    min_scale: 1.0
    pad_mode: zeros
hardware:
  platform: kaggle
  accelerator: GPU
  gpu: T4
  kaggle_image: kaggle-gpu-2026-11-08
  environment_lock: environment/versions.lock.md
outputs:
  experiment_dir: 06_Experiments/EXP0008
  checkpoint_dir: 06_Experiments/EXP0008/notebook/checkpoints
  checkpoint_manifest: 07_Checkpoints/EXP0008_checkpoints.md
  artifact_dataset_slug: isthisdeception/thesis-exp0008-artifacts
git_commit: 004fba1
timestamp: "2026-11-26T19:45:00+06:00"
'''
for cand in [
    Path('/kaggle/input/datasets/isthisdeception/ds0001-pp0001/DS0001_PP0001'),
    Path('/kaggle/input/ds0001-pp0001/DS0001_PP0001'),
]:
    if cand.exists():
        CONFIG_YAML = CONFIG_YAML.replace('/kaggle/input/datasets/isthisdeception/ds0001-pp0001/DS0001_PP0001', cand.as_posix())
        print('Resolved DS0001 mount:', cand)
        break

if not (REPO / '06_Experiments' / EXP / 'metrics' / 'run_summary.json').exists() and not (REPO / '06_Experiments' / EXP / 'notebook' / 'checkpoints' / 'checkpoint_manifest.json').exists():
    _pref = 'vit_base_patch16_224.augreg2_in21k_ft_in1k'
    _avail = set(timm.list_models(pretrained=True))
    if _pref not in _avail:
        _hits = timm.list_models('vit_base_patch16_224.*', pretrained=True) or timm.list_models('*vit_base_patch16_224*', pretrained=True)
        assert _hits, 'No pretrained vit_base_patch16_224 found in timm!'
        CONFIG_YAML = CONFIG_YAML.replace(_pref, _hits[0])
        _pref = _hits[0]
    CONFIG.parent.mkdir(parents=True, exist_ok=True)
    CONFIG.write_text(CONFIG_YAML, encoding='utf-8')
    print('Wrote self-bootstrapped config:', CONFIG, '| backbone:', _pref)

ds_reg = REPO / '03_Datasets/metadata/dataset_registry.csv'
if 'EXP0008' not in ds_reg.read_text(encoding='utf-8'):
    with ds_reg.open('a', encoding='utf-8') as f:
        f.write('DS0001,v1.0,PP0001,DS0001_PP0001_SPLIT0001,EXP0008,MODEL0004,Registry Agent,ready,2026-11-26,2026-11-26\n')

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ('GITHUB_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try:
            os.environ[_k] = _s.get_secret(_k)
            print(_k, 'loaded')
        except Exception:
            print(_k, 'not set (optional)')
except ImportError:
    pass

import torch, fastai
assert torch.cuda.is_available(), 'GPU not visible! Set Accelerator to GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__, '| FastAI:', fastai.__version__)

sys.path.insert(0, str(AUTOMATION))
from config_validation import validate_config_file
from training import run_experiment
validate_config_file(CONFIG).raise_if_failed()
print('Config validation PASS:', CONFIG)

## A. Train ViT-B/16 (`EXP0008`) & Export `val` / `test` Predictions

In [ ]:
summary = run_experiment(CONFIG)
print('best_epoch:', summary['best_epoch'], '| training_seconds:', summary['training_seconds'])
print('Test metrics:', {k: round(summary['metrics']['test'][k], 4) for k in ('roc_auc', 'pr_auc', 'balanced_accuracy', 'f1', 'mcc', 'fnr', 'fpr', 'ece')})

## B. Automatic Cross-Dataset Evaluation (`DS0002` DiFF, `DS0003` 140k, `DS0004` Synthbuster, `DS0005` FairFace)
Runs automatically for every external dataset attached to this notebook.

In [ ]:
import json, pandas as pd, yaml
from generalization.config import load_config as load_gen_config
from generalization.inference import run as run_gen_inference
from generalization.cross_dataset import dataset_frame, group_frame, relation_frame, reference_frame
from robustness.inference import load_model, sha256_file

exp_dir = REPO / '06_Experiments' / EXP
export_pkl = exp_dir / 'notebook' / 'checkpoints' / f'{MODEL_ID}_export.pkl'
thresh_val = float(json.loads((exp_dir / 'metrics' / 'threshold.json').read_text())['value'])

val_df = pd.read_csv(exp_dir / 'predictions' / 'val_predictions.csv')
test_df = pd.read_csv(exp_dir / 'predictions' / 'test_predictions.csv')
def _to_ref(df, sp):
    return pd.DataFrame({
        'id': df['content_hash'],
        'processed_path': df['processed_path'],
        'split': sp,
        'y_true': (df['class_label'] == 'fake').astype(int),
        'y_prob': df['prob_positive'],
        'y_pred': df['pred_positive'],
        'threshold': thresh_val,
        'generator': df['generator'],
    })
ref_csv = REPO / '08_Evaluation' / EXP / 'predictions.csv'
ref_csv.parent.mkdir(parents=True, exist_ok=True)
pd.concat([_to_ref(val_df, 'val'), _to_ref(test_df, 'test')], ignore_index=True).to_csv(ref_csv, index=False)

ext_candidates = [
    ('DS0002', 'DiFF official test', '03_Datasets/splits/DS0002_PP0002_SPLIT0001/assignments.csv', 'test', [
        '/kaggle/input/datasets/isthisdeception/ds0002-pp0002/DS0002_PP0002',
        '/kaggle/input/ds0002-pp0002/DS0002_PP0002',
    ], None, {}),
    ('DS0003', '140k Real and Fake Faces', '03_Datasets/splits/DS0003_PP0003_SPLIT0001/assignments.csv', 'test', [
        '/kaggle/input/datasets/isthisdeception/ds0003-pp0003/DS0003_PP0003',
        '/kaggle/input/ds0003-pp0003/DS0003_PP0003',
    ], None, {'stylegan': 'related'}),
    ('DS0004', 'Synthbuster', '03_Datasets/splits/DS0004_PP0006_SPLIT0001/assignments.csv', 'test', [
        '/kaggle/input/datasets/isthisdeception/ds0004-pp0006/DS0004_PP0006',
        '/kaggle/input/ds0004-pp0006/DS0004_PP0006',
    ], None, {'stable-diffusion-1-3': 'related', 'stable-diffusion-1-4': 'related', 'stable-diffusion-2': 'related', 'stable-diffusion-xl': 'related'}),
    ('DS0005', 'FairFace', '03_Datasets/splits/DS0005_PP0005_SPLIT0001/assignments.csv', 'test', [
        '/kaggle/input/datasets/isthisdeception/ds0005-pp0005/DS0005_PP0005',
        '/kaggle/input/ds0005-pp0005/DS0005_PP0005',
    ], 10000, {}),
]
attached_ds = []
for name, desc, assn, part, roots, cap, rel in ext_candidates:
    hit = next((r for r in roots if Path(r).exists() or Path(r).parent.exists()), None)
    if hit:
        attached_ds.append({'name': name, 'description': desc, 'assignments': assn, 'partition': part, 'processed_root': hit, 'max_images': cap, 'relation': rel})

if attached_ds:
    ds1_root = yaml.safe_load(CONFIG.read_text())['dataset']['processed_root']
    gen_dir = REPO / '08_Evaluation' / EVAL_ID / 'generalization'
    gen_dir.mkdir(parents=True, exist_ok=True)
    gen_cfg_dict = {
        'evaluation_id': EVAL_ID,
        'experiment_id': EXP,
        'seed': 42,
        'model': {
            'model_id': MODEL_ID,
            'export_name': export_pkl.name,
            'export_sha256': sha256_file(export_pkl),
            'positive_class': 'fake',
            'search': [str(export_pkl.parent)],
        },
        'reference': {
            'name': f'DS0001 SPLIT0001 test ({EXP})',
            'predictions': f'08_Evaluation/{EXP}/predictions.csv',
            'split': 'test',
            'processed_root': ds1_root,
        },
        'training_generators': ['cips', 'face_synthetics', 'gansformer', 'lama', 'mat', 'projected_gan', 'sfhq', 'stable_diffusion', 'stargan', 'stylegan1', 'stylegan2', 'stylegan3', 'taming_transformer'],
        'inference': {'image_size': 224, 'resize_interpolation': 'bilinear', 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'batch_size': 128, 'num_workers': 4, 'sanity_images': 200, 'sanity_atol': 0.002},
        'datasets': attached_ds,
    }
    gen_cfg_path = gen_dir / 'config.yaml'
    gen_cfg_path.write_text(yaml.safe_dump(gen_cfg_dict, sort_keys=False), encoding='utf-8')
    gcfg = load_gen_config(gen_cfg_path)
    model, vocab = load_model(export_pkl, 'cuda')
    frames = run_gen_inference(gcfg, REPO, model, vocab, gen_dir, Path(ds1_root), device='cuda')
    ref_f = reference_frame(gcfg, REPO)
    ds_f = dataset_frame(frames, thresh_val, ref_f)
    grp_f = group_frame(frames, thresh_val, ref_f, gcfg)
    rel_f = relation_frame(grp_f)
    ds_f.to_csv(gen_dir / 'cross_dataset_metrics.csv', index=False)
    grp_f.to_csv(gen_dir / 'cross_dataset_by_generator.csv', index=False)
    rel_f.to_csv(gen_dir / 'cross_dataset_by_relation.csv', index=False)
    print(ds_f[['dataset', 'n', 'fnr', 'fpr', 'balanced_accuracy', 'roc_auc']].to_string(index=False))
else:
    print('No external DS0002-DS0005 datasets attached; skipping cross-dataset step.')

## C. Package Downloadable `EXP0008_results_bundle.zip` (+ Optional GitHub Sync)

In [ ]:
import zipfile
from datetime import datetime

bundle_path = Path(f'/kaggle/working/{EXP}_results_bundle.zip')
collect_Targets = [
    f'06_Experiments/{EXP}/config.yaml',
    f'06_Experiments/{EXP}/logs',
    f'06_Experiments/{EXP}/metrics',
    f'06_Experiments/{EXP}/learning_curves',
    f'06_Experiments/{EXP}/predictions',
    f'06_Experiments/{EXP}/notebook/checkpoints/checkpoint_manifest.json',
    f'07_Checkpoints/{EXP}_checkpoints.md',
    f'08_Evaluation/{EXP}/predictions.csv',
    f'08_Evaluation/{EVAL_ID}/generalization',
]
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in collect_Targets:
        p = REPO / rel
        if p.is_file():
            zf.write(p, arcname=rel)
        elif p.is_dir():
            for sub in sorted(p.rglob('*')):
                if sub.is_file() and sub.suffix not in {'.pth', '.pkl', '.pt'}:
                    zf.write(sub, arcname=sub.relative_to(REPO).as_posix())
print(f'READY TO DOWNLOAD: {bundle_path} ({bundle_path.stat().st_size / 1024:.1f} KB)')

if os.environ.get('GITHUB_TOKEN') and (AUTOMATION / PLAN).exists():
    push_branch = f"kaggle/{EXP.lower()}-{datetime.now():%Y%m%d-%H%M}"
    subprocess.run([sys.executable, '-m', 'kaggle_sync', '--plan', PLAN, 'push-metadata', '--branch', push_branch, '--commit-message', f'sync: {EXP} results from Kaggle'], cwd=AUTOMATION, check=False)
    print('Attempted GitHub branch sync to:', push_branch)